# Session 1: from rain to river
**A. Real-world problem.** A storm drops the same 60 mm of rain on a town's hills twice: once in
1 hour, once spread over 6 hours. In which case does the river rise higher? The answer is not
obvious, and it decides how high a bridge or a flood wall must be.

**Plan.** (1) split rain into soaking-in and run-off, (2) turn run-off depth into a flow in
m3/s, (3) let the catchment gather the water into the river with a delay, (4) compare a short
intense storm with a long gentle one.

*Educational model.* Real flood forecasts come from official agencies, not from this lesson.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

AREA = 20.0   # catchment area in km2 (a made-up valley)
K = 2.0       # catchment delay in hours: the bigger K, the slower the river rises

## Step 1: some rain soaks in
The ground can swallow water at a rate `f` (mm per hour), like a sponge. Rain that falls faster
than `f` cannot soak in, so the excess runs off over the surface:

    run-off rate = rain intensity - f      (never below zero)

Rain of 10 mm/h on ground that swallows 5 mm/h gives 5 mm/h of run-off. Rain of 4 mm/h
gives none.

In [ ]:
def runoff_rate(i, f):
    """Run-off rate in mm/h from rain intensity i (mm/h) and soak-in rate f (mm/h)."""
    return max(i - f, 0.0)

## Step 2: run-off depth becomes a flow
1 mm of water over 1 km2 is 0.001 m * 1 000 000 m2 = 1000 m3. So 1 mm/h over 1 km2 is
1000 m3 per hour = 1000 / 3600 = 0.2778 m3/s. Hence

    flow (m3/s) = run-off rate (mm/h) * area (km2) / 3.6

In [ ]:
def to_flow(rate_mm_h, area_km2):
    """Flow in m3/s from a depth rate (mm/h) falling on an area (km2)."""
    return rate_mm_h * area_km2 / 3.6

## Step 3: the catchment gathers the water
Run-off does not reach the river at once: it must trickle down slopes and gullies. The simplest
picture is a tank (a *linear reservoir*) with a leak. Water runs in at rate `q_in`; the tank
drains at `storage / K`. Then the river flow is `storage / K`.

For a storm that gives constant run-off flow `q` for `D` hours the river flow rises as
`q (1 - exp(-t/K))`, reaches `q (1 - exp(-D/K))` at the end of the storm, then falls as
`exp(-(t-D)/K)`. A storm shorter than K never lets the river reach `q`.

In [ ]:
def storm_peak(P, D, f, area=AREA, k=K):
    """Peak river flow (m3/s) for P mm of rain falling evenly over D hours."""
    i = P / D
    q = to_flow(runoff_rate(i, f), area)
    return q * (1.0 - np.exp(-D / k))


def hydrograph(P, D, f, t, area=AREA, k=K):
    """River flow (m3/s) at times t (hours) after the storm starts, exact solution."""
    t = np.asarray(t, dtype=float)
    q = to_flow(runoff_rate(P / D, f), area)
    qpk = q * (1.0 - np.exp(-D / k))
    return np.where(t <= D, q * (1.0 - np.exp(-t / k)), qpk * np.exp(-(t - D) / k))


def simulate(P, D, f, t_end=24.0, dt=0.005, area=AREA, k=K):
    """Same flow by stepping the tank forward in small time steps (a check on the formulas)."""
    n = int(round(t_end / dt))
    q = to_flow(runoff_rate(P / D, f), area)
    s, out = 0.0, np.zeros(n + 1)
    for j in range(n):
        t = j * dt
        s += dt * ((q if t < D else 0.0) - s / k)   # storage in (m3/s * h) units
        out[j + 1] = s / k
    return np.arange(n + 1) * dt, out


def runoff_volume(P, D, f, area=AREA):
    """Run-off volume in m3."""
    return runoff_rate(P / D, f) * D * area * 1000.0

## Step 4: same rain, different storm
Both storms drop 60 mm. The soak-in rate is f = 5 mm/h, the catchment is 20 km2, K = 2 h.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    f = 5.0
    print("1 mm/h on 1 km2 =", round(to_flow(1.0, 1.0), 4), "m3/s")
    t = np.linspace(0, 16, 400)
    fig, ax = plt.subplots(figsize=(6.4, 4))
    for D, col in ((1.0, "tab:red"), (6.0, "tab:blue")):
        ax.plot(t, hydrograph(60.0, D, f, t), color=col,
                label=f"60 mm in {D:g} h: peak {storm_peak(60.0, D, f):.1f} m3/s")
        print(f"60 mm in {D:g} h -> peak {storm_peak(60.0, D, f):.1f} m3/s,",
              f"run-off {runoff_volume(60.0, D, f) / 1000:.0f} thousand m3")
    ax.set_xlabel("time since the storm began (h)")
    ax.set_ylabel("river flow (m3/s)")
    ax.set_title("Same 60 mm of rain: short and intense beats long and gentle")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "two_storms.png", dpi=120)

    Ds = np.linspace(0.5, 12, 60)
    fig2, ax2 = plt.subplots(figsize=(6.4, 4))
    for f2, col in ((0.0, "tab:green"), (5.0, "tab:orange"), (10.0, "tab:purple")):
        ax2.plot(Ds, [storm_peak(60.0, d, f2) for d in Ds], color=col, label=f"soak-in {f2:g} mm/h")
    ax2.set_xlabel("storm duration D (h), total rain fixed at 60 mm")
    ax2.set_ylabel("peak river flow (m3/s)")
    ax2.set_title("Longer storms of the same depth give smaller peaks")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "peak_vs_duration.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change P, D and f in `lesson.py` and predict the peak before you run it. Check your prediction with `storm_peak`.
2. **Stretch.** Write `longest_safe_storm(P, f)` that finds the longest duration (in 0.5 h steps up to 12 h) for which the peak still exceeds 100 m3/s.
3. **Explain.** Explain why the peak falls when the same rain is spread over a longer time, using the words intensity, soak-in and delay.

## Assessment
- **A. Conceptual.** Why can a storm with less total rain sometimes cause a bigger flood than a storm with more rain?
- **B. Numerical.** 60 mm of rain falls in 2 h on the 20 km2 catchment (f = 5 mm/h, K = 2 h). Find the river peak and check the units.
- **C. Graph and simulation interpretation.** In the graph of peak against duration (rain fixed at 60 mm), what do the 0 and 10 mm/h curves tell you at D = 6 h? What happens as D gets small?
- **D. Design and optimisation challenge.** Design: land use changes can raise f from 5 to 10 mm/h (for example, more open soil and trees). For the 60 mm, 1 h storm, does that bring the peak below 100 m3/s? What else would you change, and what do you assume?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("unit factor: 1 mm/h on 1 km2 is 1000 m3/h = 0.2778 m3/s",
                   abs(L.to_flow(1.0, 1.0) - 1000.0 / 3600.0) < 1e-9 and abs(L.to_flow(1.0, 1.0) - 0.2778) < 1e-4))
    checks.append(("rain slower than the soak-in rate gives no run-off", L.runoff_rate(4.0, 5.0) == 0.0))
    checks.append(("rain 10 mm/h on ground that swallows 5 mm/h gives 5 mm/h", L.runoff_rate(10.0, 5.0) == 5.0))
    # worked example: 60 mm in 1 h, f = 5 mm/h, 20 km2, K = 2 h
    q = L.to_flow(L.runoff_rate(60.0, 5.0), 20.0)
    pk = L.storm_peak(60.0, 1.0, 5.0)
    checks.append((f"worked example: run-off flow 305.6 m3/s and river peak 120.2 m3/s (got {q:.1f}, {pk:.1f})",
                   abs(q - 305.6) < 0.05 and abs(pk - 120.2) < 0.05 and abs(1 - np.exp(-0.5) - 0.3935) < 5e-5))
    q6 = L.to_flow(L.runoff_rate(10.0, 5.0), 20.0)
    pk6 = L.storm_peak(60.0, 6.0, 5.0)
    checks.append((f"hand calculation: the 6 h storm gives 27.78 m3/s * 0.9502 = 26.4 m3/s (got {pk6:.1f})",
                   abs(q6 - 27.78) < 0.005 and abs(1 - np.exp(-3) - 0.9502) < 5e-5 and abs(pk6 - 26.4) < 0.05))
    checks.append(("short intense storm beats long gentle storm with the same rain depth", pk > pk6))
    checks.append(("run-off volumes: 55 mm gives 1100 thousand m3, 30 mm gives 600 thousand m3",
                   abs(L.runoff_volume(60.0, 1.0, 5.0) - 1.1e6) < 1 and abs(L.runoff_volume(60.0, 6.0, 5.0) - 6.0e5) < 1))
    t, qs = L.simulate(60.0, 1.0, 5.0)
    checks.append(("time stepping agrees with the exact formula (peak within 0.5 percent)",
                   abs(qs.max() - pk) / pk < 0.005))
    checks.append(("all the run-off water reaches the river (mass kept, within 0.1 percent)",
                   abs(qs.sum() * 0.005 * 3600.0 - L.runoff_volume(60.0, 1.0, 5.0)) / 1.1e6 < 1e-3))
    checks.append(("ground that swallows everything gives zero flood", L.storm_peak(60.0, 6.0, 10.0) == 0.0))
    ds = np.linspace(0.5, 12, 40)
    pks = [L.storm_peak(60.0, d, 5.0) for d in ds]
    checks.append(("peak falls steadily as the same rain is spread over longer durations", bool(np.all(np.diff(pks) < 0))))
    checks.append(("exact hydrograph is continuous at the end of the storm",
                   abs(float(L.hydrograph(60.0, 1.0, 5.0, 1.0 + 1e-9)) - pk) < 1e-5))
    got = [L.storm_peak(60.0, 2.0, 5.0), L.storm_peak(60.0, 0.5, 5.0), L.storm_peak(60.0, 1.0, 0.0),
           L.storm_peak(60.0, 6.0, 0.0), L.storm_peak(60.0, 1.0, 10.0)]
    checks.append(("more peaks: 60 mm in 2 h gives 87.8, in 0.5 h gives 141.3; with f = 0 the 1 h and 6 h storms give 131.2 and 52.8; f = 10 turns the 1 h peak into 109.3 m3/s",
                   np.allclose(got, [87.8, 141.3, 131.2, 52.8, 109.3], atol=0.05)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
